In [1]:
"""
File: train_emotion_classifier.py
Author: Octavio Arriaga
Email: arriaga.camargo@gmail.com
Github: https://github.com/oarriaga
Description: Train emotion classification model
"""

from keras.callbacks import CSVLogger, ModelCheckpoint, EarlyStopping
from keras.callbacks import ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator

from models.cnn import mini_XCEPTION
from utils.datasets_edited import DataManager
from utils.datasets_edited import split_data
# from utils.datasets_edited import
from utils.preprocessor import preprocess_input

from math import floor

from keras.models import load_model

import os
import numpy as np
import cv2
import pandas as pd

# parameters
batch_size = 32
# num_epochs = 10000
num_epochs = 2
input_shape = (64, 64, 1)
validation_split = .2
verbose = 1
num_classes = 7
patience = 50
base_path = '../trained_models/emotion_models/'

# data generator
data_generator = ImageDataGenerator(
                        featurewise_center=False,
                        featurewise_std_normalization=False,
                        rotation_range=10,
                        width_shift_range=0.1,
                        height_shift_range=0.1,
                        zoom_range=.1,
                        horizontal_flip=True)

# TRAIN DATA
def train(dataset, split, img_shape, train_data, val_data, model):
    # pass dataset name
    dataset_name = dataset
    # callbacks
    log_file_path = base_path + dataset_name + "_" + \
                    str(img_shape) +"_"+ str(split)+'_emotion_training.log'
    csv_logger = CSVLogger(log_file_path, append=False)
    early_stop = EarlyStopping('val_loss', patience=patience)
    reduce_lr = ReduceLROnPlateau('val_loss', factor=0.1,
                                patience=int(patience/4), verbose=1)
    trained_models_path = base_path + dataset_name + '_mini_XCEPTION'
    model_names = trained_models_path + '.{epoch:02d}-{val_acc:.2f}.hdf5'
    model_checkpoint = ModelCheckpoint(model_names, 'val_loss', verbose=1,
                                                    save_best_only=True)
    callbacks = [model_checkpoint, csv_logger, early_stop, reduce_lr]

    # loading dataset
    faces, emotions = train_data
    model.fit_generator(data_generator.flow(faces, emotions,
                                            batch_size),
                        steps_per_epoch=len(faces) / batch_size,
                        epochs=num_epochs, verbose=1, callbacks=callbacks,
                        validation_data=val_data)
    
    return model_names
    ### TRAINING

# split data according to class and percentage
# input: faces_subset, labels_subset, split_percentage
# output: train_data, val_data, test_data 
def split_data(x, y, split):
    num_samples = len(x)
    num_val_samples = int(round(split*num_samples))
    num_train_samples = int(round((1 - split*2)*num_samples))
#     print((1-split*2)*num_samples)
#     print(split*num_samples)
    train_x = x[:num_train_samples]
    train_y = y[:num_train_samples]
#     print("---------train size: "+str(len(train_x)))
    val_x = x[num_train_samples:num_train_samples+num_val_samples]
    val_y = y[num_train_samples:num_train_samples+num_val_samples]
#     print("---------val size: "+str(len(val_x)))
    test_x = x[num_train_samples+num_val_samples:]
    test_y = y[num_train_samples+num_val_samples:]
#     print("---------test size: "+str(len(test_x)))
    train_data = (train_x, train_y)
    val_data = (val_x, val_y)
    test_data = (test_x, test_y)
    return train_data, val_data, test_data

# LOAD IMAGES AND LABELS IF CSV EXISTS
def load_CSV(csv_file, img_size):
    data = pd.read_csv(csv_file)
    pixels = data['face'].tolist()
    # width, height = 64, 64
    faces = []
    for pixel_sequence in pixels:
        face = [int(pixel) for pixel in pixel_sequence.split(' ')]
        face = np.asarray(face).reshape(img_size[0], img_size[1])
        face = cv2.resize(face.astype('uint8'), img_size)
        faces.append(face.astype('float32'))
    faces = np.asarray(faces)
    faces = np.expand_dims(faces, -1)
    emotions = pd.get_dummies(data['emotion']).as_matrix()

    return faces, emotions

# SHUFFLE FACES AND EMOTION ARRAYS SIMULTANEOUSLY
# source: https://stackoverflow.com/questions/4601373/better-way-to-shuffle-two-numpy-arrays-in-unison
def shuffle_in_unison(a, b):
    assert len(a) == len(b)
    shuffled_a = np.empty(a.shape, dtype=a.dtype)
    shuffled_b = np.empty(b.shape, dtype=b.dtype)
    permutation = np.random.permutation(len(a))
    for old_index, new_index in enumerate(permutation):
        shuffled_a[new_index] = a[old_index]
        shuffled_b[new_index] = b[old_index]
    return (shuffled_a, shuffled_b)

def get_labels(dataset_name):
    if dataset_name in ['FER2013', 'ExpW', 'ExpWCSV']:
        return {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy',
                4: 'sad', 5: 'surprise', 6: 'neutral'}
    elif dataset_name == 'imdb':
        return {0: 'woman', 1: 'man'}
    elif dataset_name in ['KDEF', 'KDEFCSV']:
        return {0: 'AN', 1: 'DI', 2: 'AF', 3: 'HA', 4: 'SA', 5: 'SU', 6: 'NE'}
    elif dataset_name in ['AffectNet', 'AffectNetCSV']:
        return {0: 'neutral', 1: 'happy', 2: 'sad', 3: 'surprise', 
                4: 'fear', 5: 'disgust', 6: 'anger'}
    else:
        raise Exception('Invalid dataset name')


def get_class_to_arg(dataset_name='FER2013'):
    if dataset_name in ['FER2013', 'ExpW', 'ExpWCSV']:
        return {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4,
                'surprise': 5, 'neutral': 6}
    elif dataset_name == 'imdb':
        return {'woman': 0, 'man': 1}
    elif dataset_name in ['KDEF', 'KDEFCSV']:
        return {'AN': 0, 'DI': 1, 'AF': 2, 'HA': 3, 'SA': 4, 'SU': 5, 'NE': 6}
    elif dataset_name in ['AffectNet', 'AffectNetCSV']:
        return {'neutral': 0, 'happy': 1, 'sad': 2, 'surprise': 3, 
                'fear': 4, 'disgust': 5, 'anger': 6}
    else:
        raise Exception('Invalid dataset name')

# DATASETS TO BE USED
datasets_root = "/home/ubuntu/Datasets"
datasets = ['AffectNet', 'FER2013', 'KDEF', 'ExpW']
# datasets = ['KDEF', 'ExpW']
input_shape = (64, 64, 1)
save_to_csv = []
split = 0.1
# csv_output = "collected_datasets.csv"

dataset_dict = {}


# LOAD DATASETS INTO DICTIONARY
# ----dictionary structure: {<dataset_name>:[faces, emotions, model_created]}
print("Loading datasets...")
for i, dataset in enumerate(datasets):
    # load/create csv
    csv_name = str(dataset) + "_" + str(input_shape[0]) + ".csv"
    try:
        csv_file = open(os.path.join(datasets_root, dataset, csv_name))
    except:
        print("["+str(dataset)+"] File "+csv_name+" does not exist. Loading and converting dataset.")
        path = os.path.join(datasets_root, dataset)
        csv_path = os.path.join(datasets_root, dataset, csv_name)
        data_loader = DataManager(dataset,
                                  image_size=input_shape[:2],
                                  dataset_path=path,
                                  csv_output=csv_path)
        faces, emotions = data_loader.get_data()
    else:
        print("["+str(dataset)+"] File "+csv_name+" exists. Loading file...")
        faces, emotions = load_CSV(csv_file, input_shape[:2])

    # shuffle
    print("["+str(dataset)+"] Shuffling images and labels simultaneously...")
    (faces, emotions) = shuffle_in_unison(faces,
                                        emotions)
    print("["+str(dataset)+"] Creating miniXception model...")
    dataset_dict[dataset] = [faces,
                             emotions,
                             mini_XCEPTION(input_shape, num_classes)]
    
print("Finished loading datasets.\n")

# TESTING SECTION
# for i, dataset in enumerate(datasets):
# load model
# open file
# for all data inputted to model
# get classification
# print actual then predicted on file
# close filie


Using TensorFlow backend.


Loading datasets...
[AffectNet] File AffectNet_64.csv exists. Loading file...


/home/ubuntu/miniconda3/envs/capstone_2/lib/python3.6/site-packages/ipykernel_launcher.py:113: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.


[AffectNet] Shuffling images and labels simultaneously...
[AffectNet] Creating miniXception model...
[FER2013] File FER2013_64.csv exists. Loading file...
[FER2013] Shuffling images and labels simultaneously...
[FER2013] Creating miniXception model...
[KDEF] File KDEF_64.csv exists. Loading file...
[KDEF] Shuffling images and labels simultaneously...
[KDEF] Creating miniXception model...
[ExpW] File ExpW_64.csv exists. Loading file...
[ExpW] Shuffling images and labels simultaneously...
[ExpW] Creating miniXception model...
Finished loading datasets.



In [5]:
import gc
fer_path = "/home/ubuntu/Repos/face_classification/trained_models/undersampled"
# 'AffectNet', 'FER2013', 'KDEF', 'ExpW'
fer_models = ["AffectNet_mini_XCEPTION.100-1.00.hdf5",
              "FER2013_mini_XCEPTION.118-1.00.hdf5",
              "KDEF_mini_XCEPTION.158-1.00.hdf5",
              "ExpW_mini_XCEPTION.36-1.00.hdf5"]
# fer_models = ["KDEF_mini_XCEPTION.158-1.00.hdf5",
#               "ExpW_mini_XCEPTION.36-1.00.hdf5"]
# model_name = ""
for i, fer_model in enumerate(fer_models):
#     print(fer_model)
#     emotion_model_path = os.path.join(fer_path, fer_model)
#     emotion_labels = get_labels(datasets[i])
#     # Load models

#     emotion_classifier = load_model(emotion_model_path, compile=False)
#     emotion_target_size = emotion_classifier.input_shape[1:3]

#     for j, test_dataset in enumerate(datasets):
#         if j == i:
#             continue
#         print("Current: ("+datasets[i]+") ("+datasets[j]+")")
#         f = open(datasets[i]+"_"+datasets[j]+".txt", 'w')

#         for k, image in enumerate(dataset_dict[datasets[j]][0]):

#             try:
#                 gray_face = cv2.resize(image, (emotion_target_size))
#             except:
#                 continue

#             gray_face = preprocess_input(gray_face, True)
#             gray_face = np.expand_dims(gray_face, 0)
#             gray_face = np.expand_dims(gray_face, -1)

#             emotion_prediction = emotion_classifier.predict(gray_face)
#             emotion_probability = np.max(emotion_prediction)  # returns probability value
#             emotion_label_arg = np.argmax(emotion_prediction)  # returns index of highest probability value
#             emotion_text = emotion_labels[emotion_label_arg]

#             actual_label = int(np.argwhere(dataset_dict[datasets[j]][1][k]==1))
#             pred_label = emotion_label_arg
#             f.write(str(pred_label)+","+str(actual_label)+"\n")
#         f.close()

        print(dataset_dict[datasets[i]][2])
        gc.collect()